# Column with node-to-surface constraints — v5

Same model as v4, but now the static analysis is captured as a **time series** so the viewer shows the load being applied progressively.

## What's new in v5

`ops.integrator('LoadControl', 0.1)` runs the load in **10 pseudo-time steps** (λ = 0.1, 0.2, ..., 1.0). In v4 we only extracted the final step. Here we capture the displacement field after every `ops.analyze(1)` call and pass the whole series to `Results.from_fem(steps=[...])`. The viewer picks this up automatically and enables its time-step slider — drag it to scrub through the loading history, or watch the viewer animate between snapshots.

Each step dict has the shape:

```python
{
    "time": λ,                              # 0.1, 0.2, ..., 1.0
    "point_data": {
        "Displacement": disp_at_step,       # (N, 3)
        "|U|":          u_mag_at_step,      # (N,)
        "Ux":           ux_at_step,         # (N,)
        "Uy":           uy_at_step,         # (N,)
        "Uz":           uz_at_step,         # (N,)
    },
}
```

The scalar range on each contour field auto-adjusts as you scrub, so bending modes that are invisible at step 1 (small load, small displacement) become legible when you zoom the color bar — and symmetrical fields like `Ux` (Poisson contraction) scale linearly with λ as expected.

In [1]:
from apeGmsh import apeGmsh, Part, Results
from apeGmsh import FEMData
from apeGmsh.sections import W_solid
import numpy as np
from pathlib import Path

In [2]:
m1 = apeGmsh(model_name='beam_column', verbose=False)
m1.begin()

column = m1.sections.W_solid(bf=150, tf=20, h=300, tw=10, length=2000, label='column')

m1.model.selection.select_volumes().to_physical(name='pg_column')

base = m1.model.geometry.add_point(x=0, y=0, z=0,    lc=100, label='base')
top  = m1.model.geometry.add_point(x=0, y=0, z=2000, lc=100, label='top')

m1.constraints.node_to_surface('base', column.labels.start_face)
m1.constraints.node_to_surface('top',  column.labels.end_face)

m1.loads.point(
    target='top',
    force_xyz=[0, 1000, 0],
    moment_xyz=[0, 0, 0],
)

m1.mesh.sizing.set_global_size(100)
m1.mesh.generation.generate(dim=3)

fem = m1.mesh.queries.get_fem_data(remove_orphans=True)

m1.end()

## OpenSees model (same as v4)

- `ndf = 6` global, solid column nodes overridden to `ndf = 3`.
- Grounded helper node at `GROUND_TAG` fixed in 6 DOFs, connected to the base reference point via a stiff 6-DOF `zeroLength` spring.
- `constraints('Penalty')` because the phantom nodes are daisy-chained.

In [3]:
import openseespy.opensees as ops

ops.wipe()
ops.model('basic', '-ndm', 3, '-ndf', 6)

ops.timeSeries('Linear', 1)

# --- Nodes -----------------------------------------------------------------
for nid, xyz in fem.nodes.get(target='pg_column'):
    ops.node(nid, *xyz, '-ndf', 3)

ref = fem.nodes.get(target=['base', 'top'])
for nid, xyz in ref:
    ops.node(nid, *xyz)

for nid, xyz in fem.nodes.constraints.phantom_nodes():
    ops.node(nid, *xyz)

# --- Grounded helper node + zeroLength spring at the base ------------------
base_id = int(next(iter(fem.nodes.get_ids(target='base'))))
base_xyz = next(xyz for nid, xyz in ref if nid == base_id)

GROUND_TAG = 10_000
ops.node(GROUND_TAG, *base_xyz)
ops.fix(GROUND_TAG, 1, 1, 1, 1, 1, 1)

K_SPRING = 1e14
for i in range(6):
    ops.uniaxialMaterial('Elastic', 100 + i, K_SPRING)

ZL_TAG = 20_000
ops.element('zeroLength', ZL_TAG, GROUND_TAG, base_id,
            '-mat', 100, 101, 102, 103, 104, 105,
            '-dir', 1, 2, 3, 4, 5, 6)

# --- Material + tet elements -----------------------------------------------
E  = 200e3
nu = 0.3
ops.nDMaterial('ElasticIsotropic', 1, E, nu)

for group in fem.elements.get(element_type='tet4'):
    for eid, conn in group:
        ops.element('FourNodeTetrahedron', int(eid), *[int(n) for n in conn], 1)

# --- MP constraints --------------------------------------------------------
for master, slaves in fem.nodes.constraints.rigid_link_groups():
    for slave in slaves:
        ops.rigidLink('beam', int(master), int(slave))

for pair in fem.nodes.constraints.equal_dofs():
    ops.equalDOF(int(pair.master_node), int(pair.slave_node), *pair.dofs)

# --- Loads -----------------------------------------------------------------
ops.pattern('Plain', 1, 1)
for load in fem.nodes.loads.by_kind('nodal'):
    fx, fy, fz = load.force_xyz  or (0.0, 0.0, 0.0)
    mx, my, mz = load.moment_xyz or (0.0, 0.0, 0.0)
    ops.load(int(load.node_id), fx, fy, fz, mx, my, mz)

# --- Analysis --------------------------------------------------------------
ops.constraints('Penalty', 1e15, 1e15)
ops.numberer('RCM')
ops.system('UmfPack')
ops.test('NormDispIncr', 1.0e-6, 10)
ops.algorithm('Linear')
ops.integrator('LoadControl', 0.1)
ops.analysis('Static')

## Step-wise analysis with displacement snapshots

After each `ops.analyze(1)` call we query `ops.nodeDisp` for every node in `fem.nodes.ids` and append a step dict. The pseudo-time stored in each step is the current load factor (`ops.getTime()`), so the slider labels are meaningful.

In [4]:
node_ids_list = [int(nid) for nid in fem.nodes.ids]
n_nodes = len(node_ids_list)
n_steps = 10

steps: list[dict] = []

for i in range(n_steps):
    ok = ops.analyze(1)
    if ok != 0:
        print(f'Step {i+1}: failed ({ok})')
        break

    t = float(ops.getTime())

    disp = np.zeros((n_nodes, 3), dtype=np.float64)
    for j, nid in enumerate(node_ids_list):
        d = ops.nodeDisp(nid)
        disp[j, 0] = d[0]
        disp[j, 1] = d[1]
        disp[j, 2] = d[2]

    u_mag = np.linalg.norm(disp, axis=1)

    steps.append({
        'time': t,
        'point_data': {
            'Displacement': disp,
            '|U|':          u_mag,
            'Ux':           disp[:, 0].copy(),
            'Uy':           disp[:, 1].copy(),
            'Uz':           disp[:, 2].copy(),
        },
    })

    print(f'Step {i+1:2d}  λ = {t:.2f}   |U|_max = {u_mag.max():.4e}')

print(f'\nCaptured {len(steps)} steps.')

Step  1  λ = 0.10   |U|_max = 7.9292e-03
Step  2  λ = 0.20   |U|_max = 1.5857e-02
Step  3  λ = 0.30   |U|_max = 2.3787e-02
Step  4  λ = 0.40   |U|_max = 3.1714e-02
Step  5  λ = 0.50   |U|_max = 3.9645e-02
Step  6  λ = 0.60   |U|_max = 4.7572e-02


Step  7  λ = 0.70   |U|_max = 5.5504e-02
Step  8  λ = 0.80   |U|_max = 6.3431e-02
Step  9  λ = 0.90   |U|_max = 7.1359e-02
Step 10  λ = 1.00   |U|_max = 7.9288e-02

Captured 10 steps.


## Base reaction at the final step

Expected from equilibrium: `Fy = -1000 N`, `Mx = +2·10⁶ N·mm`.

In [5]:
ops.reactions()

rxn = ops.nodeReaction(GROUND_TAG)

print(f'nodeReaction(ground = {GROUND_TAG}):')
print(f'  Fx = {rxn[0]:+.4e}   Fy = {rxn[1]:+.4e}   Fz = {rxn[2]:+.4e}')
print(f'  Mx = {rxn[3]:+.4e}   My = {rxn[4]:+.4e}   Mz = {rxn[5]:+.4e}')
print(f'\nExpected:  Fy = -1.0000e+03   Mx = +2.0000e+06')

nodeReaction(ground = 10000):
  Fx = +6.3563e-05   Fy = -1.0000e+03   Fz = -6.9247e-05
  Mx = +2.0000e+06   My = +6.3709e-02   Mz = +3.7051e+00

Expected:  Fy = -1.0000e+03   Mx = +2.0000e+06


## Build the time-series Results and launch the viewer

Passing `steps=[...]` instead of `point_data=...` produces a time-series result. The viewer writes a PVD + 10 VTU files to a temp directory and opens them together; the time-step slider in the controls panel is automatically enabled and labelled with the load factor.

Drag the slider to scrub through the loading history. Combine with the Deformation toggle to see the column progressively bend into its final shape.

In [6]:
# --- LEGACY-API-MIGRATED ---
# Materialize ``steps`` into ``(T, N)`` arrays for ``NativeWriter``.
# Vector fields (shape ``(N, 3)``) split into per-axis scalar components.
_legacy_components = {}
for _legacy_cname, _legacy_cval0 in steps[0]["point_data"].items():
    _legacy_arr0 = np.asarray(_legacy_cval0)
    if _legacy_arr0.ndim == 2 and _legacy_arr0.shape[1] in (2, 3):
        for _legacy_i, _legacy_ax in enumerate(["x", "y", "z"][: _legacy_arr0.shape[1]]):
            _legacy_components[f"{_legacy_cname}_{_legacy_ax}"] = np.stack(
                [np.asarray(_s["point_data"][_legacy_cname])[:, _legacy_i]
                 for _s in steps], axis=0,
            )
    else:
        _legacy_components[_legacy_cname] = np.stack(
            [np.asarray(_s["point_data"][_legacy_cname]) for _s in steps],
            axis=0,
        )
_legacy_time = np.array([_s["time"] for _s in steps], dtype=float)
# Determine node count from the data, then pick node_ids that match.
_legacy_first = next(iter(steps[0]["point_data"].values()))
_legacy_N = int(np.asarray(_legacy_first).shape[0])
if None:
    _legacy_node_ids = np.asarray(
        fem.nodes.get_ids(pg=(None)[0] if None else None), dtype=np.int64,
    )
    if _legacy_node_ids.size != _legacy_N:
        # Falls back to all nodes if the pg cardinality doesn't match
        # the per-step data (covers cases where the user collected
        # disp on a different subset than the legacy ``include_pgs``).
        _legacy_node_ids = np.asarray(fem.nodes.ids, dtype=np.int64)
else:
    _legacy_node_ids = np.asarray(fem.nodes.ids, dtype=np.int64)
if _legacy_node_ids.size != _legacy_N:
    raise RuntimeError(
        f"node_ids has {_legacy_node_ids.size} entries but step data "
        f"has {_legacy_N} — adjust the source of node_ids in this cell."
    )
from pathlib import Path as _LegacyPath
from apeGmsh.results.writers import NativeWriter as _LegacyNativeWriter
_legacy_path = _LegacyPath(f"{'column_nts_v5'}_legacy.h5")
if _legacy_path.exists():
    _legacy_path.unlink()
with _LegacyNativeWriter(_legacy_path) as _legacy_nw:
    _legacy_nw.open(fem=fem)
    _legacy_sid = _legacy_nw.begin_stage(name='column_nts_v5', kind="static", time=_legacy_time)
    _legacy_nw.write_nodes(
        _legacy_sid, "partition_0",
        node_ids=_legacy_node_ids, components=_legacy_components,
    )
    _legacy_nw.end_stage()
results = Results.from_native(_legacy_path, fem=fem)

print(f'Results: {len(steps)} steps, {n_nodes} nodes')
print(f'Time range: [{steps[0]["time"]:.2f}, {steps[-1]["time"]:.2f}]')
results

Results: 10 steps, 646 nodes
Time range: [0.10, 1.00]


<>:24: SyntaxWarning: 'NoneType' object is not subscriptable; perhaps you missed a comma?
<>:24: SyntaxWarning: 'NoneType' object is not subscriptable; perhaps you missed a comma?
C:\Users\nmora\AppData\Local\Temp\ipykernel_45100\3400969028.py:24: SyntaxWarning: 'NoneType' object is not subscriptable; perhaps you missed a comma?
  fem.nodes.get_ids(pg=(None)[0] if None else None), dtype=np.int64,


Results: column_nts_v5_legacy.h5
  FEM: 646 nodes, 3684 elements (snapshot_id=6bb161648ef68e796d35a7c20ced744f)
  Stages (1):
    - stage_0 (column_nts_v5): steps=10, kind=static

In [7]:
results.viewer(blocking=False)

<Popen: returncode: None args: ['C:\\Users\\nmora\\venv\\opensees_venv\\Scri...>

## Cantilever closed-form sanity check (final step)

Strong-axis bending of the W section (`Ix = bf·H³/12 − (bf − tw)·h³/12`, `H = 340 mm`):

- `δ_EB = P·L³ / (3·E·Ix)`  (Euler-Bernoulli)
- `δ_T  = δ_EB + P·L / (G·As)`, `As ≈ tw·h`  (Timoshenko, adds shear)

In [8]:
top_id = int(next(iter(fem.nodes.get_ids(target='top'))))
tip_disp = ops.nodeDisp(top_id)

bf, tf, h_web, tw, L = 150.0, 20.0, 300.0, 10.0, 2000.0
H  = 2 * tf + h_web
Ix = (bf * H**3) / 12 - ((bf - tw) * h_web**3) / 12
P  = 1000.0

delta_EB    = P * L**3 / (3 * E * Ix)
G           = E / (2 * (1 + nu))
As          = tw * h_web
delta_shear = P * L / (G * As)
delta_T     = delta_EB + delta_shear

print(f'Top node {top_id} displacement (final step):')
print(f'  ux = {tip_disp[0]:+.4e}   uy = {tip_disp[1]:+.4e}   uz = {tip_disp[2]:+.4e}')
print(f'\nCantilever closed-form:')
print(f'  delta_EB         = {delta_EB:.4e} mm')
print(f'  delta_Timoshenko = {delta_T:.4e} mm')
print(f'  FEM uy           = {tip_disp[1]:+.4e} mm')
print(f'  FEM / EB         = {tip_disp[1] / delta_EB:.4f}')
print(f'  FEM / Timoshenko = {tip_disp[1] / delta_T:.4f}')

Top node 34 displacement (final step):
  ux = -1.6530e-04   uy = +7.8710e-02   uz = +2.4277e-06

Cantilever closed-form:
  delta_EB         = 7.5629e-02 mm
  delta_Timoshenko = 8.4295e-02 mm
  FEM uy           = +7.8710e-02 mm
  FEM / EB         = 1.0407
  FEM / Timoshenko = 0.9337


## Tip displacement progression

Sanity check: since the problem is linear, `u_y(top)` should scale exactly linearly with the load factor. Plotting `|U|_max` across steps confirms this and cross-checks the stored time-series against the final tip displacement.

In [9]:
lambdas = np.array([s['time'] for s in steps])
umax    = np.array([s['point_data']['|U|'].max() for s in steps])

print('  step    λ        |U|_max')
print('  ----  ------  -----------')
for i, (lam, u) in enumerate(zip(lambdas, umax), 1):
    print(f'  {i:>4}  {lam:6.2f}  {u:11.4e}')

ratio = umax[-1] / lambdas[-1] if lambdas[-1] else 0
residuals = np.abs(umax - ratio * lambdas)
print(f'\nLinearity check:')
print(f'  |U|_max / λ (const):   {ratio:.4e}')
print(f'  max abs residual:      {residuals.max():.3e}')
print(f'  (a linear system should give residual ~ 0)')

  step    λ        |U|_max
  ----  ------  -----------
     1    0.10   7.9292e-03
     2    0.20   1.5857e-02
     3    0.30   2.3787e-02
     4    0.40   3.1714e-02
     5    0.50   3.9645e-02
     6    0.60   4.7572e-02
     7    0.70   5.5504e-02
     8    0.80   6.3431e-02
     9    0.90   7.1359e-02
    10    1.00   7.9288e-02

Linearity check:
  |U|_max / λ (const):   7.9288e-02
  max abs residual:      2.236e-06
  (a linear system should give residual ~ 0)


## 9. Capture results — manual + DomainCapture paths

Two ways to produce a native-HDF5 results file consumable by the
apeGmsh ``ResultsViewer``:

1. **Manual path** — query the live OpenSees domain post-analysis,
   open a ``NativeWriter``, and write nodal displacements yourself.
   Good for one-shot snapshots and post-hoc diagnostics.
2. **DomainCapture path** — declare what to capture with
   ``Recorders().nodes(...)``, hand the spec to a ``DomainCapture``
   context, and call ``cap.step(t=...)`` after each ``ops.analyze``
   (the helper does it for you). Scales to multi-stage, transient,
   modal, and multi-recorder runs.

Both produce a file that ``Results.from_native(path).viewer()`` can
open. The viewer launch is gated on ``APEGMSH_SKIP_VIEWER`` so this
notebook is safe to run under nbconvert / CI.


In [10]:
# --- EOS-WIRING-V1 ---
# Manual path: pull displacements off the live domain, write h5 yourself.
from pathlib import Path
import numpy as np
from apeGmsh.results.writers import NativeWriter

results_manual = Path("example_column_nodeToSurface_v5_manual.h5")
if results_manual.exists():
    results_manual.unlink()

_n = len(fem.nodes.ids)
_ux = np.array([ops.nodeDisp(int(nid), 1) for nid in fem.nodes.ids])
_uy = np.array([ops.nodeDisp(int(nid), 2) for nid in fem.nodes.ids])
_uz = np.array([ops.nodeDisp(int(nid), 3) for nid in fem.nodes.ids])
_components = {
    "displacement_x": _ux.reshape(1, _n),
    "displacement_y": _uy.reshape(1, _n),
    "displacement_z": _uz.reshape(1, _n),
}

with NativeWriter(results_manual) as _nw:
    _nw.open(fem=fem)
    _sid = _nw.begin_stage(name="static", kind="static", time=np.array([1.0]))
    _nw.write_nodes(
        _sid, "partition_0",
        node_ids=np.asarray(fem.nodes.ids, dtype=np.int64),
        components=_components,
    )
    _nw.end_stage()

print(f"manual -> {results_manual} ({results_manual.stat().st_size/1024:.1f} KB)")


manual -> example_column_nodeToSurface_v5_manual.h5 (443.1 KB)


In [11]:
# DomainCapture path: declarative recorders, capture during analyze.
from apeGmsh.solvers.Recorders import Recorders
from apeGmsh.results.capture._domain import DomainCapture

recs = Recorders()
recs.nodes(components="displacement")
recs.nodes(components="reaction_force")
spec = recs.resolve(fem, ndm=3, ndf=6)

results_capture = Path("example_column_nodeToSurface_v5_capture.h5")
if results_capture.exists():
    results_capture.unlink()

with DomainCapture(spec, results_capture, fem, ndm=3, ndf=6) as cap:
    cap.begin_stage("run", kind="static")
    # --- copied from cell 4 ---
    import openseespy.opensees as ops

    ops.wipe()
    ops.model('basic', '-ndm', 3, '-ndf', 6)

    ops.timeSeries('Linear', 1)

    # --- Nodes -----------------------------------------------------------------
    for nid, xyz in fem.nodes.get(target='pg_column'):
        ops.node(nid, *xyz, '-ndf', 3)

    ref = fem.nodes.get(target=['base', 'top'])
    for nid, xyz in ref:
        ops.node(nid, *xyz)

    for nid, xyz in fem.nodes.constraints.phantom_nodes():
        ops.node(nid, *xyz)

    # --- Grounded helper node + zeroLength spring at the base ------------------
    base_id = int(next(iter(fem.nodes.get_ids(target='base'))))
    base_xyz = next(xyz for nid, xyz in ref if nid == base_id)

    GROUND_TAG = 10_000
    ops.node(GROUND_TAG, *base_xyz)
    ops.fix(GROUND_TAG, 1, 1, 1, 1, 1, 1)

    K_SPRING = 1e14
    for i in range(6):
        ops.uniaxialMaterial('Elastic', 100 + i, K_SPRING)

    ZL_TAG = 20_000
    ops.element('zeroLength', ZL_TAG, GROUND_TAG, base_id,
                '-mat', 100, 101, 102, 103, 104, 105,
                '-dir', 1, 2, 3, 4, 5, 6)

    # --- Material + tet elements -----------------------------------------------
    E  = 200e3
    nu = 0.3
    ops.nDMaterial('ElasticIsotropic', 1, E, nu)

    for group in fem.elements.get(element_type='tet4'):
        for eid, conn in group:
            ops.element('FourNodeTetrahedron', int(eid), *[int(n) for n in conn], 1)

    # --- MP constraints --------------------------------------------------------
    for master, slaves in fem.nodes.constraints.rigid_link_groups():
        for slave in slaves:
            ops.rigidLink('beam', int(master), int(slave))

    for pair in fem.nodes.constraints.equal_dofs():
        ops.equalDOF(int(pair.master_node), int(pair.slave_node), *pair.dofs)

    # --- Loads -----------------------------------------------------------------
    ops.pattern('Plain', 1, 1)
    for load in fem.nodes.loads.by_kind('nodal'):
        fx, fy, fz = load.force_xyz  or (0.0, 0.0, 0.0)
        mx, my, mz = load.moment_xyz or (0.0, 0.0, 0.0)
        ops.load(int(load.node_id), fx, fy, fz, mx, my, mz)

    # --- Analysis --------------------------------------------------------------
    ops.constraints('Penalty', 1e15, 1e15)
    ops.numberer('RCM')
    ops.system('UmfPack')
    ops.test('NormDispIncr', 1.0e-6, 10)
    ops.algorithm('Linear')
    ops.integrator('LoadControl', 0.1)
    ops.analysis('Static')
    # --- copied from cell 6 ---
    node_ids_list = [int(nid) for nid in fem.nodes.ids]
    n_nodes = len(node_ids_list)
    n_steps = 10

    steps: list[dict] = []

    for i in range(n_steps):
        ok = ops.analyze(1)
        cap.step(t=ops.getTime())
        if ok != 0:
            print(f'Step {i+1}: failed ({ok})')
            break

        t = float(ops.getTime())

        disp = np.zeros((n_nodes, 3), dtype=np.float64)
        for j, nid in enumerate(node_ids_list):
            d = ops.nodeDisp(nid)
            disp[j, 0] = d[0]
            disp[j, 1] = d[1]
            disp[j, 2] = d[2]

        u_mag = np.linalg.norm(disp, axis=1)

        steps.append({
            'time': t,
            'point_data': {
                'Displacement': disp,
                '|U|':          u_mag,
                'Ux':           disp[:, 0].copy(),
                'Uy':           disp[:, 1].copy(),
                'Uz':           disp[:, 2].copy(),
            },
        })

        print(f'Step {i+1:2d}  λ = {t:.2f}   |U|_max = {u_mag.max():.4e}')

    print(f'\nCaptured {len(steps)} steps.')
    cap.end_stage()

print(f"capture -> {results_capture} ({results_capture.stat().st_size/1024:.1f} KB)")


Step  1  λ = 0.10   |U|_max = 7.9292e-03


Step  2  λ = 0.20   |U|_max = 1.5857e-02
Step  3  λ = 0.30   |U|_max = 2.3787e-02


Step  4  λ = 0.40   |U|_max = 3.1714e-02
Step  5  λ = 0.50   |U|_max = 3.9645e-02
Step  6  λ = 0.60   |U|_max = 4.7572e-02
Step  7  λ = 0.70   |U|_max = 5.5504e-02


Step  8  λ = 0.80   |U|_max = 6.3431e-02
Step  9  λ = 0.90   |U|_max = 7.1359e-02


Step 10  λ = 1.00   |U|_max = 7.9288e-02

Captured 10 steps.
capture -> example_column_nodeToSurface_v5_capture.h5 (730.7 KB)


In [12]:
# Open the captured run in the apeGmsh ResultsViewer (subprocess,
# non-blocking). Set APEGMSH_SKIP_VIEWER=1 to skip in headless / CI.
import os
from apeGmsh.results import Results
results = Results.from_native(results_capture)
if os.environ.get("APEGMSH_SKIP_VIEWER"):
    print("[skip viewer] APEGMSH_SKIP_VIEWER set")
else:
    handle = results.viewer(blocking=False)
    print(f"viewer pid: {handle.pid}  -- close window to exit.")


[skip viewer] APEGMSH_SKIP_VIEWER set
